# FHIRPathを試してみよう！～Advanced編～

## 📓テーマ

Fairway Hospital の看護師 Maria Carter は、FHIR リポジトリに保存されている HbA1c 値を抽出・分析することで、2 型糖尿病のリスクがある患者を早期に特定し、生活習慣改善などの予防的介入を行いたいと考えています。

メモ：HbA1c は、過去1〜2か月程度の平均的な血糖状態を示す検査値です。

Maria は、HbA1c 値が糖尿病予備軍の範囲にある患者を確認するだけでなく、患者ごとの HbA1c の変化を追跡し、値が上昇傾向にある患者を優先的なフォローアップ対象として抽出したいと考えています。

FHIR の SearchParameter を使った検索と、FHIRPath を利用して、対象患者を検索・分析してみましょう。

## 事前準備

確認：100Set フォルダ以下にあるサンプルリソース（json）の IRIS へのロードは済んでいますか？

これからロードする場合は、[事前準備：2.サンプルデータのロード](./README.md#2-サンプルデータのロード) をご覧ください。


### 手順1：必要なパッケージ、FHIRリポジトリのエンドポイントの設定を行います。

In [ ]:
from fhirpathpy import evaluate
import matplotlib.pyplot as plt
import requests
from requests.auth import HTTPBasicAuth

### 手順2：HbA1cの検査結果を入手します。

📝メモ：どのようなリソースが何件登録されているかについては、[Try-FHIRPath-Advanced.http](./Try-FHIRPath-Advanced.http)で事前に確認できます。

💡ヒント：データ量が多いので`_count=50` を指定しています。ページ送りしながら全データを入手していきます。

In [ ]:
url = "http://localhost/irishealth/csp/healthshare/r4fhirnamespace/fhir/r4/Observation?code=4548-4&value-quantity=ge5.7&value-quantity=le6.4&_count=50"
headers = {
    "Accept": "*/*",
    "content-type": "application/fhir+json",
    "Accept-Encoding": "gzip, deflate, br",
    "Prefer": "return=representation"
}
response = requests.get(url, headers=headers, auth=HTTPBasicAuth('SuperUser', 'SYS'))
bundle=response.json()

#### PatientのリソースID／Observation.effectiveDateTime／Observation.valueQuantity.valueを抽出し、まとめる

In [ ]:
#PatientのリソースID
subjects = evaluate(bundle,"Bundle.entry.resource.subject.reference",[])
#Observation.effectiveDateTime
dates = evaluate(bundle,"Bundle.entry.resource.effectiveDateTime",[])
#Observation.valueQuantity.value
values = evaluate(bundle,"Bundle.entry.resource.valueQuantity.value",[])

print(subjects)
print(dates)
print(values)

In [ ]:
results = []

for subject, date, value in zip(subjects, dates, values):
    results.append({
        "patient": subject,
        "date": date,
        "hba1c": float(value)
    })

print(results)

#### ページ送りをして残りのページのデータを抽出する

上記セルの中身をページ送りの指定分だけ実行します。

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from fhirpathpy import evaluate

headers = {
    "Accept": "application/fhir+json",
    "content-type": "application/fhir+json"
}

auth = HTTPBasicAuth("SuperUser", "SYS")

url = (
    "http://localhost/irishealth/csp/healthshare/r4fhirnamespace/fhir/r4/"
    "Observation?code=4548-4&value-quantity=ge5.7&value-quantity=le6.4&_count=50"
)

all_results = []

while url:
    response = requests.get(url, headers=headers, auth=auth)
    response.raise_for_status()

    bundle = response.json()

    subjects = evaluate(bundle, "Bundle.entry.resource.subject.reference", [])
    dates = evaluate(bundle, "Bundle.entry.resource.effectiveDateTime", [])
    values = evaluate(bundle, "Bundle.entry.resource.valueQuantity.value", [])

    for subject, date, value in zip(subjects, dates, values):
        all_results.append({
            "patient": subject,
            "date": date,
            "hba1c": float(value)
        })

    # next link を探す
    next_links = [
        link["url"]
        for link in bundle.get("link", [])
        if link.get("relation") == "next"
    ]

    url = next_links[0] if next_links else None

print(f"全体の件数は、{len(all_results)}")
# 先頭5件を表示
all_results[:5]

#### 患者ごとの HbA1c の変化を追跡し、値が上昇傾向にある患者を調べます。

🔍以下手順で調べます。

同じ患者のHbA1cを日付順に並べる

　↓

一番古い値と一番新しい値を比べる

　↓

新しい値の方が高ければ「上昇傾向」

In [ ]:
# 患者ごとにデータをまとめる
patient_data = {}

for result in all_results:
    patient = result["patient"]

    if patient not in patient_data:
        patient_data[patient] = []

    patient_data[patient].append(result)

patient_data


# 上昇傾向の患者を探す
increasing_patients = []

for patient, records in patient_data.items():

    # 日付順に並べる
    records_sorted = sorted(records, key=lambda x: x["date"])

    oldest = records_sorted[0]
    newest = records_sorted[-1]

    # 一番古い値と一番新しい値を比較
    if newest["hba1c"] > oldest["hba1c"]:
        increasing_patients.append({
            "patient": patient,
            "old_date": oldest["date"],
            "old_hba1c": oldest["hba1c"],
            "new_date": newest["date"],
            "new_hba1c": newest["hba1c"],
            "difference": newest["hba1c"] - oldest["hba1c"]
        })

print(f"上昇傾向の患者数: {len(increasing_patients)}")
increasing_patients[:10]

#### pandasを使って少し見やすくする

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

df_increasing = pd.DataFrame(increasing_patients)
df_increasing

#### おまけ：振れ幅上位3件の患者データのHbA1cの推移をグラフにする

In [ ]:
# 上昇幅が大きい順に並べてTop3を取る
top3_patients = sorted(
    increasing_patients,
    key=lambda x: x["difference"],
    reverse=True
)[:3]

top3_patient_ids = [p["patient"] for p in top3_patients]
top3_patient_ids

##### 並べ替えなどが簡単になるので、pandas の DataFrameを使う

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(all_results)

# 日付変換（UTCに統一）
df["date"] = pd.to_datetime(df["date"], format="mixed", utc=True)

for patient_id in top3_patient_ids:
    patient_df = df[df["patient"] == patient_id].sort_values("date")

    plt.plot(
        patient_df["date"],
        patient_df["hba1c"],
        marker="o",
        label=patient_id
    )

plt.xlabel("Date")
plt.ylabel("HbA1c")
plt.title("HbA1c Trend for Top 3 Increasing Patients")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.show()